# Single-turn Error Analysis (Phase 3)

Purpose: inspect the full LLM-seeded Phase 3 run (`results/single_turn_llm_full_experiment/full_llm_single_turn.csv`) for edge cases, classifier confidence, and emotion/strategy mismatches. Findings are logged as Markdown blocks alongside each test.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

BASE = Path("../results/single_turn_llm_full_experiment")
csv_path = BASE / "full_llm_single_turn.csv"
ci_path = BASE / "full_llm_single_turn_ci.csv"

assert csv_path.exists(), f"Missing run CSV at {csv_path}"
df = pd.read_csv(csv_path)
ci_df = pd.read_csv(ci_path) if ci_path.exists() else None

df.head()

,intended_emotion,seed_text,seed_emotion_detected,seed_confidence,strategy,style_modifier,strategy_reply,followup_reply,followup_emotion,followup_confidence
0,anger,I can't believe you did that! It's completely ...,anger,0.910539,Validate,concise and emotionally attuned,It sounds like you're really hurt and frustrat...,"Yes, I'm hurt and frustrated! It's like you ju...",anger,0.957437
1,anger,I can't believe you did that! How could you be...,surprise,0.774749,Validate,concise and emotionally attuned,It sounds like you're really frustrated and hu...,Frustrated doesn't even begin to cover it! You...,disgust,0.544470
2,anger,I can't believe you would do something like th...,anger,0.450175,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,"Hurt or not, what you did was wrong, and I won...",anger,0.434986
3,anger,I can't believe you would do something so self...,anger,0.952229,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,Hurt and frustrated is putting it mildly! It’s...,anger,0.678558
4,anger,I can't believe you did that! You have no idea...,anger,0.806245,Validate,concise and emotionally attuned,It sounds like you're feeling really hurt and ...,I can't believe you just brushed off my anger ...,anger,0.956390


In [4]:
# Seed and follow-up alignment
seed_match = (df["intended_emotion"] == df["seed_emotion_detected"]).rename("seed_match")
follow_match = (df["intended_emotion"] == df["followup_emotion"]).rename("follow_match")
seed_match_rate = seed_match.mean()
follow_match_rate = follow_match.mean()

df_with_flags = df.assign(seed_match=seed_match.values, follow_match=follow_match.values)
by_strategy_follow = df_with_flags.groupby("strategy")["follow_match"].mean().sort_values(ascending=False)
by_emotion_follow = df_with_flags.groupby("intended_emotion")["follow_match"].mean().sort_values(ascending=False)

display(by_strategy_follow.to_frame("followup_match_rate"))
display(by_emotion_follow.to_frame("followup_match_rate"))

display(Markdown(
    f"**Alignment summary:** Seed match rate = {seed_match_rate:.2%}; "
    f"follow-up match rate = {follow_match_rate:.2%}. Baseline follow-up match = {by_strategy_follow.get('baseline', float('nan')):.2%}."
))

KeyError: 'Column not found: None'

In [ ]:
# Confidence diagnostics
seed_conf = df["seed_confidence"]
follow_conf = df["followup_confidence"]

summary = pd.DataFrame({
    "metric": ["seed_confidence", "followup_confidence"],
    "mean": [seed_conf.mean(), follow_conf.mean()],
    "median": [seed_conf.median(), follow_conf.median()],
    "p10": [seed_conf.quantile(0.10), follow_conf.quantile(0.10)],
    "p90": [seed_conf.quantile(0.90), follow_conf.quantile(0.90)],
})

low_conf = df[df["followup_confidence"] < 0.5]
display(summary)
display(Markdown(
    f"**Confidence summary:** Follow-up mean={follow_conf.mean():.2f}, median={follow_conf.median():.2f}, "
    f"10th percentile={follow_conf.quantile(0.10):.2f}; low-confidence (<0.5) count={len(low_conf)} / {len(df)}."
))

In [ ]:
# Sample slices for manual inspection
def show_samples(title, frame, n=5, random_state=0):
    if frame.empty:
        display(Markdown(f"**{title}:** none found."))
        return
    cols = [
        "intended_emotion",
        "strategy",
        "seed_emotion_detected",
        "seed_text",
        "strategy_reply",
        "followup_emotion",
        "followup_reply",
        "followup_confidence",
    ]
    display(Markdown(f"**{title} (showing up to {n}):**"))
    display(frame[cols].sample(min(n, len(frame)), random_state=random_state))

mismatch = df[df["followup_emotion"] != df["intended_emotion"]]
low_conf_follow = df[df["followup_confidence"] < 0.5]
neutral_drift = df[(df["intended_emotion"] == "neutral") & (df["followup_emotion"] != "neutral")]

show_samples("Follow-up mismatches", mismatch)
show_samples("Low-confidence follow-ups", low_conf_follow)
show_samples("Neutral starts that shifted", neutral_drift)


In [ ]:
# Wilson CI sanity check (optional)
if ci_df is not None:
    top = (
        ci_df.sort_values("proportion", ascending=False)
        .groupby(["intended_emotion", "strategy"])
        .head(1)[["intended_emotion", "strategy", "target_emotion", "proportion", "ci_low", "ci_high"]]
    )
    display(top.head(10))
    display(Markdown("**CI check:** Top outcome per (emotion, strategy) with 95% Wilson intervals shown above."))
else:
    display(Markdown("**CI check:** CI file not found."))

## Findings (auto-logged)
- Seed match rate and follow-up match rates are printed above; look for strategies with notably lower alignment.
- Low-confidence follow-ups table surfaces cases for manual reading.
- Mismatch/neutral-drift samples provide qualitative evidence for the report.